In [1]:
"""
CHƯƠNG TRÌNH THỐNG KÊ DỮ LIỆU CHUYÊN SÂU CHO 3 BIẾN [Date, Ticker, Close]
Học phần: Phân tích Dữ liệu Kinh doanh / Tài chính Định lượng (PRJ05)
Tập dữ liệu: 17_2521001081_PRJ05_NguyenPhiLamNguyen.csv
"""

import sys
import os
import warnings
import pandas as pd
import numpy as np

# Đảm bảo hiển thị tiếng Việt UTF-8 chuẩn trên Terminal Windows
if sys.platform == 'win32':
    sys.stdout.reconfigure(encoding='utf-8')

warnings.filterwarnings('ignore')

# 1. ĐỌC DỮ LIỆU GỐC
file_path = 'DS_17_2521001081_PRJ05_NguyenPhiLamNguyen.csv'
if not os.path.exists(file_path):
    file_path = '17_2521001081_PRJ05_NguyenPhiLamNguyen.csv'
if not os.path.exists(file_path):
    file_path = r'C:\Users\LENOVO\Downloads\DS_17_2521001081_PRJ05_NguyenPhiLamNguyen.csv'

print(f"🔄 Đang tải dữ liệu từ: {file_path} ...")
df_raw = pd.read_csv(file_path, low_memory=False)
total_raw_rows = len(df_raw)

# Lọc riêng 3 biến mục tiêu
df_3var = df_raw[['Date', 'Ticker', 'Close']].copy()

print("\n" + "="*85)
print("PHẦN 1: THỐNG KÊ ĐỘ SẠCH DỮ LIỆU BAN ĐẦU CHO 3 BIẾN [Date, Ticker, Close]")
print("="*85)

# 1.1. Thống kê biến Date
date_raw = df_3var['Date'].astype(str).str.strip()
date_missing = df_3var['Date'].isna() | date_raw.isin(['', 'nan', 'NaN', 'None', 'null', '?', '-'])
date_dirty_symbol = date_raw.isin(['-', '?']).sum()

# 1.2. Thống kê biến Ticker
ticker_raw = df_3var['Ticker'].astype(str)
ticker_stripped = ticker_raw.str.strip()
ticker_spaces = (ticker_raw != ticker_stripped).sum()

# 1.3. Thống kê biến Close
close_raw = df_3var['Close'].astype(str).str.strip()
close_missing = df_3var['Close'].isna() | close_raw.isin(['', 'nan', 'NaN', 'None', 'null', '?', '-'])
units = ['USD', 'VND', 'kg', '%', 'đ']
close_with_units = close_raw.str.contains('|'.join(units), regex=True, na=False).sum()
close_with_comma = close_raw.str.contains(',', na=False).sum()

# Kiểm toán số lượng & tỷ lệ
audit_table = [
    {
        'Tên biến': 'Date (Thời gian)',
        'Số dòng thiếu': f"{date_missing.sum():,}",
        'Tỷ lệ thiếu (%)': f"{date_missing.sum()/total_raw_rows*100:.2f}%",
        'Lỗi điển hình': f"Dính dấu '?' và '-' ({date_dirty_symbol:,} dòng)"
    },
    {
        'Tên biến': 'Ticker (Mã CP)',
        'Số dòng thiếu': "0",
        'Tỷ lệ thiếu (%)': "0.00%",
        'Lỗi điển hình': f"Dính khoảng trắng ({ticker_spaces:,} dòng) gây phân mảnh 1,509 mã ảo"
    },
    {
        'Tên biến': 'Close (Giá chốt phiên)',
        'Số dòng thiếu': f"{close_missing.sum():,}",
        'Tỷ lệ thiếu (%)': f"{close_missing.sum()/total_raw_rows*100:.2f}%",
        'Lỗi điển hình': f"Dính đơn vị {units} ({close_with_units:,} dòng), dấu phẩy ({close_with_comma:,} dòng)"
    }
]
print(pd.DataFrame(audit_table).to_string(index=False))

# ==============================================================================
# PHẦN 2: QUY TRÌNH TIỀN XỬ LÝ & LÀM SẠCH 3 BIẾN
# ==============================================================================
print("\n" + "="*85)
print("PHẦN 2: THỰC HIỆN TIỀN XỬ LÝ & LÀM SẠCH (DATA CLEANING PIPELINE)")
print("="*85)

# Bước 1: Chuẩn hóa Ticker
df_3var['Ticker'] = df_3var['Ticker'].astype(str).str.strip()

# Bước 2: Chuẩn hóa Date
date_clean_str = df_3var['Date'].astype(str).str.replace('/', '-', regex=False).str.strip()
date_clean_str = date_clean_str.replace(['-', '?', 'nan', 'NaN', 'None', ''], np.nan)
df_3var['Date'] = pd.to_datetime(date_clean_str, errors='coerce')

# Bước 3: Chuẩn hóa số Close (xóa đơn vị, dấu phẩy -> dấu chấm)
def clean_close_val(val):
    if pd.isna(val): return np.nan
    s = str(val).strip()
    for u in units:
        s = s.replace(u, '')
    s = s.replace(' ', '').replace(',', '.')
    try:
        n = float(s)
        return n if n > 0 else np.nan  # Loại bỏ cả giá trị <= 0 và mã hóa lỗi -1
    except:
        return np.nan

df_3var['Close'] = df_3var['Close'].apply(clean_close_val)

# Bước 4: Lọc các dòng có Date hợp lệ, sắp xếp thứ tự Panel Data [Ticker, Date]
df_clean = df_3var.dropna(subset=['Date']).sort_values(by=['Ticker', 'Date']).reset_index(drop=True)

# Bước 5: Lấp đầy khuyết thiếu cho Close bằng kỹ thuật Forward Fill (ffill) theo từng Ticker
df_clean['Close'] = df_clean.groupby('Ticker')['Close'].ffill()
# Loại bỏ những dòng khuyết ở đầu chuỗi không thể ffill
df_clean = df_clean.dropna(subset=['Close']).reset_index(drop=True)

print(f"✓ Tổng số quan sát ban đầu : {total_raw_rows:,} dòng")
print(f"✓ Tổng số quan sát sạch    : {len(df_clean):,} dòng")
print(f"✓ Tỷ lệ dữ liệu sử dụng    : {len(df_clean)/total_raw_rows*100:.2f}%")
print(f"✓ Số lượng mã cổ phiếu S&P 500 chuẩn: {df_clean['Ticker'].nunique()} mã")

# ==============================================================================
# PHẦN 3: BẢNG THỐNG KÊ MÔ TẢ ĐỊNH LƯỢNG CHO BIẾN GIÁ ĐÓNG CỬA (CLOSE)
# ==============================================================================
print("\n" + "="*85)
print("PHẦN 3: THỐNG KÊ MÔ TẢ ĐỊNH LƯỢNG CHO BIẾN GIÁ ĐÓNG CỬA (CLOSE)")
print("="*85)

close_series = df_clean['Close']
mean_c = close_series.mean()
std_c = close_series.std()
min_c = close_series.min()
q25_c = close_series.quantile(0.25)
med_c = close_series.median()
q75_c = close_series.quantile(0.75)
max_c = close_series.max()
iqr_c = q75_c - q25_c
skew_c = close_series.skew()
kurt_c = close_series.kurtosis()

stat_close = {
    'Chỉ tiêu thống kê': [
        'Cỡ mẫu quan sát (N)',
        'Giá trị trung bình (Mean)',
        'Độ lệch chuẩn (Std Dev)',
        'Phương sai (Variance)',
        'Giá trị nhỏ nhất (Min)',
        'Phân vị 25% (Q1)',
        'Trung vị (Median / 50%)',
        'Phân vị 75% (Q3)',
        'Giá trị lớn nhất (Max)',
        'Khoảng biến thiên (Range)',
        'Khoảng liên phân vị (IQR)',
        'Hệ số biến thiên (CV %)',
        'Hệ số bất đối xứng (Skewness)',
        'Hệ số độ nhọn (Kurtosis)'
    ],
    'Giá trị': [
        f"{len(close_series):,}",
        f"${mean_c:,.2f}",
        f"${std_c:,.2f}",
        f"{close_series.var():,.2f}",
        f"${min_c:,.2f}",
        f"${q25_c:,.2f}",
        f"${med_c:,.2f}",
        f"${q75_c:,.2f}",
        f"${max_c:,.2f}",
        f"${max_c - min_c:,.2f}",
        f"${iqr_c:,.2f}",
        f"{(std_c / mean_c)*100:.2f}%",
        f"{skew_c:.4f}",
        f"{kurt_c:.4f}"
    ],
    'Ý nghĩa kinh tế / tài chính': [
        'Tổng số ngày giao dịch của toàn bộ 503 cổ phiếu sau làm sạch',
        'Mức giá đóng cửa trung bình của thị trường',
        'Độ phân tán của thị giá xung quanh giá trị trung bình',
        'Bình phương độ lệch chuẩn đo lường biến động chung',
        'Thị giá thấp nhất từng ghi nhận trong chu kỳ',
        '25% số phiên có thị giá thấp hơn mức này',
        'Mức giá ngăn đôi phân phối (ít bị ảnh hưởng bởi ngoại lai)',
        '75% số phiên có thị giá thấp hơn mức này',
        'Thị giá đỉnh cao nhất ghi nhận trong chu kỳ',
        'Độ giãn biên độ giá cực đại (Max - Min)',
        'Độ biến động tập trung của 50% quan sát ở giữa (Q3 - Q1)',
        'Tỷ lệ rủi ro trên một đơn vị giá trị trung bình',
        '> 0: Phân phối lệch phải (phần lớn giá tập trung ở mức thấp)',
        '> 3: Đuôi dày (Fat-tail), xuất hiện nhiều biến động cực đoan'
    ]
}
print(pd.DataFrame(stat_close).to_string(index=False))

# ==============================================================================
# PHẦN 4: THỐNG KÊ THEO CHIỀU KHÔNG GIAN (TICKER) & THỜI GIAN (DATE)
# ==============================================================================
print("\n" + "="*85)
print("PHẦN 4: THỐNG KÊ CHI TIẾT THEO CHIỀU KHÔNG GIAN (TICKER) & THỜI GIAN (DATE)")
print("="*85)

# Thời gian
start_date = df_clean['Date'].min()
end_date = df_clean['Date'].max()
total_trading_days = df_clean['Date'].nunique()

print(f"1. Trục thời gian (Date Index):")
print(f"   + Ngày bắt đầu quan sát : {start_date.strftime('%Y-%m-%d')}")
print(f"   + Ngày kết thúc         : {end_date.strftime('%Y-%m-%d')}")
print(f"   + Tổng số phiên giao dịch: {total_trading_days:,} ngày")

# Số lượng quan sát của từng Ticker
obs_per_ticker = df_clean.groupby('Ticker')['Date'].count()
print(f"\n2. Phân bổ phiên giao dịch theo từng cổ phiếu (503 Tickers):")
print(f"   + Trung bình mỗi mã có  : {obs_per_ticker.mean():.1f} phiên giao dịch")
print(f"   + Mã có nhiều phiên nhất: {obs_per_ticker.max()} phiên")
print(f"   + Mã có ít phiên nhất   : {obs_per_ticker.min()} phiên")

# Top 5 cổ phiếu có thị giá trung bình cao nhất & thấp nhất
avg_price_per_ticker = df_clean.groupby('Ticker')['Close'].mean()

print("\n3. Top 5 cổ phiếu có thị giá Close trung bình cao nhất:")
for t, p in avg_price_per_ticker.sort_values(ascending=False).head(5).items():
    print(f"   - {t:<6}: ${p:,.2f}")

print("\n4. Top 5 cổ phiếu có thị giá Close trung bình thấp nhất:")
for t, p in avg_price_per_ticker.sort_values(ascending=True).head(5).items():
    print(f"   - {t:<6}: ${p:,.2f}")

# ==============================================================================
# PHẦN 5: THỐNG KÊ TỶ SUẤT SINH LỢI HÀNG NGÀY (LOG RETURNS)
# Biến số trọng tâm của bài toán tài chính tính từ [Date, Ticker, Close]
# ==============================================================================
print("\n" + "="*85)
print("PHẦN 5: THỐNG KÊ TỶ SUẤT SINH LỢI HÀNG NGÀY (DAILY LOG RETURNS - r_t)")
print("="*85)

# Tính log-return theo từng cổ phiếu: r_t = ln(Close_t / Close_{t-1})
df_clean['Log_Return'] = df_clean.groupby('Ticker')['Close'].apply(lambda x: np.log(x / x.shift(1))).reset_index(level=0, drop=True)
returns = df_clean['Log_Return'].dropna()

ret_mean = returns.mean()
ret_std = returns.std()
ret_annual_mean = ret_mean * 252 * 100
ret_annual_vol = ret_std * np.sqrt(252) * 100

stat_return = {
    'Chỉ tiêu Tỷ suất sinh lợi': [
        'Số lượng quan sát tỷ suất lợi nhuận',
        'Lợi suất trung bình ngày (Daily Mean)',
        'Độ lệch chuẩn rủi ro ngày (Daily Std)',
        'Lợi suất trung bình năm hóa (Annualized Return)',
        'Độ biến động rủi ro năm hóa (Annualized Volatility)',
        'Lợi suất ngày thấp nhất (Min Daily Return)',
        'Lợi suất ngày cao nhất (Max Daily Return)',
        'Độ lệch phân phối lợi suất (Skewness)',
        'Độ nhọn phân phối lợi suất (Kurtosis)'
    ],
    'Giá trị': [
        f"{len(returns):,}",
        f"{ret_mean*100:.4f}% / ngày",
        f"{ret_std*100:.4f}% / ngày",
        f"{ret_annual_mean:.2f}% / năm",
        f"{ret_annual_vol:.2f}% / năm",
        f"{returns.min()*100:.2f}%",
        f"{returns.max()*100:.2f}%",
        f"{returns.skew():.4f}",
        f"{returns.kurtosis():.4f}"
    ]
}
print(pd.DataFrame(stat_return).to_string(index=False))

print("\n" + "="*85)
print("✅ HOÀN TẤT BÁO CÁO THỐNG KÊ TOÀN DIỆN CHO BỘ 3 BIẾN [Date, Ticker, Close]!")
print("="*85)


SyntaxError: closing parenthesis ']' does not match opening parenthesis '{' on line 241 (1464265559.py, line 252)

In [ ]:
"""
================================================================================
DỰ ÁN TIỂU LUẬN CUỐI KỲ: PHÂN TÍCH DỮ LIỆU TÀI CHÍNH ĐỊNH LƯỢNG (PRJ05)
TẬP DỮ LIỆU S&P 500 (2021 - 2026): 17_2521001081_PRJ05_NguyenPhiLamNguyen.csv
MODULE CHUYÊN SÂU: LÀM SẠCH DỮ LIỆU (DATA CLEANING) & TRỰC QUAN HÓA (VISUALIZATION)
================================================================================
"""

import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# Đảm bảo hiển thị chuẩn tiếng Việt UTF-8
if sys.platform == 'win32':
    sys.stdout.reconfigure(encoding='utf-8')

warnings.filterwarnings('ignore')

# Cấu hình đồ họa chất lượng cao chuẩn bài báo khoa học / tiểu luận
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Calibri', 'Segoe UI']
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams['figure.titlesize'] = 14
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
CHARTS_DIR = os.path.join(BASE_DIR, 'charts')
os.makedirs(CHARTS_DIR, exist_ok=True)

# 1. ĐỌC DỮ LIỆU GỐC
file_path = 'DS_17_2521001081_PRJ05_NguyenPhiLamNguyen.csv'
if not os.path.exists(file_path):
    file_path = '17_2521001081_PRJ05_NguyenPhiLamNguyen.csv'
if not os.path.exists(file_path):
    file_path = r'C:\Users\LENOVO\Downloads\DS_17_2521001081_PRJ05_NguyenPhiLamNguyen.csv'

print(f"🔄 Đang tải dữ liệu gốc từ: {file_path} ...")
df_raw = pd.read_csv(file_path, low_memory=False)
total_raw_rows = len(df_raw)
print(f"✓ Đã nạp thành công: {total_raw_rows:,} dòng, {df_raw.shape[1]} cột.")

# ==============================================================================
# GIAI ĐOẠN 1: KIỂM TOÁN VÀ THỰC THI QUY TRÌNH LÀM SẠCH DỮ LIỆU
# ==============================================================================
print("\n" + "="*85)
print("GIAI ĐOẠN 1: KIỂM TOÁN TOÀN DIỆN & LÀM SẠCH DỮ LIỆU (DATA CLEANING PIPELINE)")
print("="*85)

units = ['USD', 'VND', 'kg', '%', 'đ']

def count_dirty(col_series):
    raw_str = col_series.astype(str).str.strip()
    missing = col_series.isna() | raw_str.isin(['', 'nan', 'NaN', 'None', 'null', '?', '-'])
    with_units = raw_str.str.contains('|'.join(units), regex=True, na=False)
    with_comma = raw_str.str.contains(',', na=False)
    invalid_chars = raw_str.isin(['?', '-', '-1', '9999'])
    return missing.sum(), (with_units | with_comma | invalid_chars).sum()

audit_records = []
cols_to_check = ['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume', 'DGS3MO']
for c in cols_to_check:
    if c in df_raw.columns:
        m_cnt, d_cnt = count_dirty(df_raw[c])
        audit_records.append({
            'Biến': c,
            'Số dòng thiếu (Missing)': m_cnt,
            'Tỷ lệ thiếu (%)': m_cnt / total_raw_rows * 100,
            'Số dòng dính lỗi rác/đơn vị': d_cnt
        })

df_audit = pd.DataFrame(audit_records)
print(df_audit.to_string(index=False))

# --- TIẾN HÀNH LÀM SẠCH ---
df_cleaned = df_raw.copy()

# A. Chuẩn hóa Ticker: Cắt sạch khoảng trắng đầu/cuối
raw_ticker_count = df_cleaned['Ticker'].nunique()
df_cleaned['Ticker'] = df_cleaned['Ticker'].astype(str).str.strip()
clean_ticker_count = df_cleaned['Ticker'].nunique()
print(f"\n1. [Ticker] Chuẩn hóa khoảng trắng: {raw_ticker_count:,} mã ảo -> {clean_ticker_count} mã S&P 500 chuẩn.")

# B. Chuẩn hóa Date: chuyển đổi định dạng và lọc bỏ ký tự rác
date_str = df_cleaned['Date'].astype(str).str.replace('/', '-', regex=False).str.strip()
date_str = date_str.replace(['-', '?', 'nan', 'NaN', 'None', ''], np.nan)
df_cleaned['Date'] = pd.to_datetime(date_str, errors='coerce')
valid_dates_cnt = df_cleaned['Date'].notna().sum()
print(f"2. [Date] Nhận diện ngày hợp lệ: {valid_dates_cnt:,} dòng (loại {total_raw_rows - valid_dates_cnt:,} dòng rác).")

# C. Hàm chuẩn hóa số thực
def clean_numeric_val(val, allow_zero=False):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    for u in units:
        s = s.replace(u, '')
    s = s.replace(' ', '').replace(',', '.')
    try:
        n = float(s)
        if n == 9999.0 or n == -1.0:
            return np.nan
        if not allow_zero and n <= 0:
            return np.nan
        return n
    except:
        return np.nan

print("3. [Numeric] Chuẩn hóa đơn vị đo lường, dấu phẩy và loại bỏ giá trị âm/mã lỗi...")
for num_col in ['Close', 'Open', 'High', 'Low']:
    if num_col in df_cleaned.columns:
        df_cleaned[num_col] = df_cleaned[num_col].apply(clean_numeric_val)

if 'Volume' in df_cleaned.columns:
    df_cleaned['Volume'] = df_cleaned['Volume'].apply(lambda x: clean_numeric_val(x, allow_zero=True))

if 'DGS3MO' in df_cleaned.columns:
    def clean_rate(v):
        n = clean_numeric_val(v, allow_zero=True)
        return n if (n is not None and not np.isnan(n) and 0 <= n <= 30) else np.nan
    df_cleaned['DGS3MO'] = df_cleaned['DGS3MO'].apply(clean_rate)

# D. Lọc dòng Date hợp lệ và thiết lập Panel Data [Ticker, Date]
df_cleaned = df_cleaned.dropna(subset=['Date']).sort_values(by=['Ticker', 'Date']).reset_index(drop=True)

# E. Xử lý ngoại lai dị thường (Outlier & Decimal shift error: dính 10x, 100x, 1000x lỗi tỷ giá/dấu chấm)
# Kỹ thuật: So sánh Close với trung vị (Median) của từng mã cổ phiếu
ticker_median = df_cleaned.groupby('Ticker')['Close'].transform('median')
outlier_mask = (df_cleaned['Close'] > 3.0 * ticker_median) | (df_cleaned['Close'] < ticker_median / 3.0)
outlier_count = outlier_mask.sum()
print(f"4. [Outliers] Phát hiện & sửa lỗi nhảy số thập phân/ngoại lai đột biến: {outlier_count:,} quan sát.")
df_cleaned.loc[outlier_mask, 'Close'] = np.nan

# F. Lấp đầy dữ liệu khuyết thiếu bằng Forward Fill (ffill) theo từng Ticker
df_cleaned['Close'] = df_cleaned.groupby('Ticker')['Close'].ffill()
df_clean_3var = df_cleaned[['Date', 'Ticker', 'Close']].dropna(subset=['Close']).reset_index(drop=True)

# G. Tính tỷ suất sinh lợi Log Return hàng ngày
df_clean_3var['Log_Return'] = df_clean_3var.groupby('Ticker')['Close'].apply(
    lambda x: np.log(x / x.shift(1))
).reset_index(level=0, drop=True)

# Lưu tập dữ liệu sạch hoàn chỉnh
cleaned_csv_path = os.path.join(BASE_DIR, 'data_cleaned_SP500.csv')
df_clean_3var.to_csv(cleaned_csv_path, index=False)

print(f"✓ Hoàn tất làm sạch: Lưu tập dữ liệu vào: {cleaned_csv_path}")
print(f"✓ Cỡ mẫu sau làm sạch: {len(df_clean_3var):,} dòng ({len(df_clean_3var)/total_raw_rows*100:.2f}%)")
print(f"✓ Giá Close trung bình mới: ${df_clean_3var['Close'].mean():,.2f}")
print(f"✓ Giá Close trung vị mới   : ${df_clean_3var['Close'].median():,.2f}")
print(f"✓ Giá Close cao nhất (NVR) : ${df_clean_3var['Close'].max():,.2f}")

# ==============================================================================
# GIAI ĐOẠN 2: THIẾT KẾ VÀ XUẤT BỘ 7 BIỂU ĐỒ TRỰC QUAN HÓA CHUẨN XUẤT BẢN
# ==============================================================================
print("\n" + "="*85)
print("GIAI ĐOẠN 2: THỰC THI TRỰC QUAN HÓA CHUYÊN SÂU (7 BIỂU ĐỒ CAO CẤP)")
print("="*85)

# Bảng màu nhận diện tài chính chuyên nghiệp
C_NAVY  = "#1B365D"
C_SLATE = "#2B547E"
C_TEAL  = "#008080"
C_CORAL = "#D9534F"
C_GOLD  = "#F0AD4E"

# ------------------------------------------------------------------------------
# BIỂU ĐỒ 1: KIỂM TOÁN CHẤT LƯỢNG DỮ LIỆU BAN ĐẦU
# ------------------------------------------------------------------------------
print("📊 [1/7] Xuất Biểu đồ 1: Thống kê tỷ lệ khuyết thiếu ban đầu...")
fig, ax = plt.subplots(figsize=(10, 5), dpi=300)
bars = ax.bar(df_audit['Biến'], df_audit['Tỷ lệ thiếu (%)'], color=C_SLATE, width=0.52, edgecolor=C_NAVY, alpha=0.9)

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 0.4, f"{yval:.2f}%", ha='center', va='bottom', fontsize=9.5, fontweight='bold', color=C_NAVY)

ax.set_title("TỶ LỆ KHUYẾT THIẾU DỮ LIỆU THEO BIẾN (DATA AUDIT TRƯỚC LÀM SẠCH)", fontsize=13, fontweight='bold', pad=15, color=C_NAVY)
ax.set_ylabel("Tỷ lệ mất mát dữ liệu (%)", fontsize=11, fontweight='bold')
ax.set_xlabel("Các thuộc tính trong tập dữ liệu S&P 500", fontsize=11, fontweight='bold')
ax.set_ylim(0, 20)
ax.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
p1 = os.path.join(CHARTS_DIR, "01_data_cleaning_audit.png")
plt.savefig(p1)
plt.close()

# ------------------------------------------------------------------------------
# BIỂU ĐỒ 2: PHÂN PHỐI GIÁ ĐÓNG CỬA (HISTOGRAM & BOXPLOT ĐA TẦNG)
# ------------------------------------------------------------------------------
print("📊 [2/7] Xuất Biểu đồ 2: Phân phối giá đóng cửa Close chuẩn hóa...")
fig, (ax_box, ax_hist) = plt.subplots(2, 1, figsize=(11, 7), sharex=True, gridspec_kw={'height_ratios': [0.22, 0.78]}, dpi=300)

c_series = df_clean_3var['Close']
mean_c = c_series.mean()
med_c = c_series.median()
q25_c = c_series.quantile(0.25)
q75_c = c_series.quantile(0.75)
p98_c = c_series.quantile(0.98)

# Lọc vùng tập trung chính 98% để hiển thị chi tiết hình dáng phân phối
sub_c = c_series[c_series <= p98_c]

sns.boxplot(x=sub_c, ax=ax_box, color="#90CAF9", flierprops=dict(marker='o', markersize=2, alpha=0.3, color=C_CORAL))
ax_box.set(xlabel='')
ax_box.set_title("PHÂN PHỐI VÀ ĐỘ LỆCH CỦA BIẾN GIÁ ĐÓNG CỬA (CLOSE PRICE DISTRIBUTION)", fontsize=13, fontweight='bold', pad=12, color=C_NAVY)

sns.histplot(sub_c, bins=60, kde=True, ax=ax_hist, color=C_SLATE, edgecolor='white', alpha=0.7)

ax_hist.axvline(mean_c, color=C_CORAL, linestyle='--', linewidth=2, label=f'Mean (Toàn bộ): ${mean_c:,.2f}')
ax_hist.axvline(med_c, color='green', linestyle='-', linewidth=2, label=f'Median: ${med_c:,.2f}')
ax_hist.axvline(q25_c, color=C_GOLD, linestyle=':', linewidth=1.5, label=f'Q1 (25%): ${q25_c:,.2f}')
ax_hist.axvline(q75_c, color=C_GOLD, linestyle=':', linewidth=1.5, label=f'Q3 (75%): ${q75_c:,.2f}')

ax_hist.set_xlabel("Giá đóng cửa Close ($ USD / cổ phiếu)", fontsize=11, fontweight='bold')
ax_hist.set_ylabel("Tần số xuất hiện (Số phiên)", fontsize=11, fontweight='bold')
ax_hist.xaxis.set_major_formatter(ticker.StrMethodFormatter('${x:,.0f}'))
ax_hist.yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))
ax_hist.legend(loc='upper right', frameon=True, facecolor='white', framealpha=0.9)
ax_hist.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
p2 = os.path.join(CHARTS_DIR, "02_close_price_distribution.png")
plt.savefig(p2)
plt.close()

# ------------------------------------------------------------------------------
# BIỂU ĐỒ 3: XU HƯỚNG GIÁ S&P 500 VÀ TOP BIG-TECH CHU KỲ 2021-2026
# ------------------------------------------------------------------------------
print("📊 [3/7] Xuất Biểu đồ 3: Diễn biến chuỗi thời gian giá cổ phiếu...")
fig, ax = plt.subplots(figsize=(12, 6), dpi=300)

market_avg = df_clean_3var.groupby('Date')['Close'].mean()
ax.plot(market_avg.index, market_avg.values, label='S&P 500 Bình quân thị trường (503 mã)', color='black', linewidth=2.5, zorder=5)

key_tickers = ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'JPM']
pal = [C_SLATE, C_TEAL, C_CORAL, C_GOLD, '#8E44AD']

for t, col in zip(key_tickers, pal):
    sub = df_clean_3var[df_clean_3var['Ticker'] == t]
    if len(sub) > 0:
        ax.plot(sub['Date'], sub['Close'], label=f'{t}', color=col, linewidth=1.6, alpha=0.85)

ax.set_title("DIỄN BIẾN THỊ GIÁ BÌNH QUÂN S&P 500 & TOP CỔ PHIẾU TIÊU BIỂU (2021 - 2026)", fontsize=13, fontweight='bold', pad=15, color=C_NAVY)
ax.set_ylabel("Giá đóng cửa ($ USD)", fontsize=11, fontweight='bold')
ax.set_xlabel("Thời gian giao dịch", fontsize=11, fontweight='bold')
ax.yaxis.set_major_formatter(ticker.StrMethodFormatter('${x:,.0f}'))
ax.legend(loc='upper left', frameon=True, ncol=3)
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
p3 = os.path.join(CHARTS_DIR, "03_market_time_series.png")
plt.savefig(p3)
plt.close()

# ------------------------------------------------------------------------------
# BIỂU ĐỒ 4: PHÂN PHỐI LỢI SUẤT HÀNG NGÀY & KIỂM ĐỊNH HIỆN TƯỢNG ĐUÔI DÀY (FAT-TAIL)
# ------------------------------------------------------------------------------
print("📊 [4/7] Xuất Biểu đồ 4: Phân phối Tỷ suất sinh lợi hàng ngày (Log Returns)...")
fig, ax = plt.subplots(figsize=(11, 6), dpi=300)

returns = df_clean_3var['Log_Return'].dropna()
clean_ret = returns[(returns >= -0.12) & (returns <= 0.12)] * 100

mu = clean_ret.mean()
sigma = clean_ret.std()

sns.histplot(clean_ret, bins=120, stat='density', color=C_SLATE, alpha=0.6, label='Phân phối lợi suất thực tế', ax=ax)

x_ax = np.linspace(-12, 12, 500)
gauss = (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x_ax - mu) / sigma)**2)
ax.plot(x_ax, gauss, color=C_CORAL, linewidth=2.5, linestyle='--', label=f'Phân phối chuẩn Gauss: N(μ={mu:.2f}%, σ={sigma:.2f}%)')

ax.set_title("PHÂN PHỐI TỶ SUẤT SINH LỢI HÀNG NGÀY (DAILY LOG RETURNS) & HIỆN TƯỢNG ĐUÔI DÀY", fontsize=13, fontweight='bold', pad=15, color=C_NAVY)
ax.set_xlabel("Tỷ suất sinh lợi ngày r_t (%)", fontsize=11, fontweight='bold')
ax.set_ylabel("Mật độ xác suất (Density)", fontsize=11, fontweight='bold')
ax.set_xlim(-10, 10)

skew_val = returns.skew()
kurt_val = returns.kurtosis()
ax.text(0.03, 0.76, f"Hệ số Skewness : {skew_val:+.4f}\nHệ số Kurtosis : {kurt_val:+.2f} (Fat-tail đỉnh nhọn đuôi dày)\nTổng quan sát  : {len(returns):,} phiên",
        transform=ax.transAxes, fontsize=10, bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor=C_SLATE, alpha=0.9))

ax.legend(loc='upper right', frameon=True)
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
p4 = os.path.join(CHARTS_DIR, "04_daily_returns_fat_tail.png")
plt.savefig(p4)
plt.close()

# ------------------------------------------------------------------------------
# BIỂU ĐỒ 5: MA TRẬN ĐÁNH ĐỔI RỦI RO - LỢI NHUẬN (RISK-RETURN TRADEOFF & SHARPE)
# ------------------------------------------------------------------------------
print("📊 [5/7] Xuất Biểu đồ 5: Đồ thị đánh đổi Rủi ro - Lợi nhuận & Tỷ số Sharpe...")
ticker_stats = []
rf_annual = 0.035  # Giả định lãi suất phi rủi ro chuẩn 3.5%

for t_name, grp in df_clean_3var.groupby('Ticker'):
    s_ret = grp['Log_Return'].dropna()
    if len(s_ret) >= 200:
        a_ret = s_ret.mean() * 252
        a_vol = s_ret.std() * np.sqrt(252)
        sh = (a_ret - rf_annual) / a_vol if a_vol > 0 else 0
        ticker_stats.append({
            'Ticker': t_name,
            'Annual_Return': a_ret * 100,
            'Annual_Volatility': a_vol * 100,
            'Sharpe': sh
        })

df_metrics = pd.DataFrame(ticker_stats)
df_metrics_plot = df_metrics[(df_metrics['Annual_Volatility'] <= 60) & (df_metrics['Annual_Return'] >= -30) & (df_metrics['Annual_Return'] <= 80)]

fig, ax = plt.subplots(figsize=(11, 7), dpi=300)
scatter = ax.scatter(df_metrics_plot['Annual_Volatility'], df_metrics_plot['Annual_Return'],
                     c=df_metrics_plot['Sharpe'], cmap='viridis', s=45, alpha=0.85, edgecolors='none')

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Tỷ số Sharpe (Sharpe Ratio)', fontsize=11, fontweight='bold')

notable_stocks = ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'JPM', 'XOM', 'TSLA', 'GOOGL', 'PG', 'LLY']
for _, row in df_metrics_plot.iterrows():
    if row['Ticker'] in notable_stocks:
        ax.annotate(row['Ticker'], (row['Annual_Volatility'], row['Annual_Return']),
                    xytext=(5, 5), textcoords='offset points', fontsize=9, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='yellow', alpha=0.75, edgecolor='gray'))

ax.axhline(0, color='red', linestyle='--', linewidth=1, alpha=0.7)
ax.set_title("MA TRẬN ĐÁNH ĐỔI RỦI RO - LỢI NHUẬN (RISK - RETURN TRADEOFF S&P 500)", fontsize=13, fontweight='bold', pad=15, color=C_NAVY)
ax.set_xlabel("Độ biến động rủi ro năm hóa (Annualized Volatility - σ %)", fontsize=11, fontweight='bold')
ax.set_ylabel("Tỷ suất sinh lợi bình quân năm hóa (Annualized Return - E(R) %)", fontsize=11, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
p5 = os.path.join(CHARTS_DIR, "05_risk_return_scatter.png")
plt.savefig(p5)
plt.close()

# ------------------------------------------------------------------------------
# BIỂU ĐỒ 6: MỨC SỤT GIẢM TỐI ĐA (MAXIMUM DRAWDOWN - MDD) THEO THỜI GIAN
# ------------------------------------------------------------------------------
print("📊 [6/7] Xuất Biểu đồ 6: Phân tích mức sụt giảm tối đa Drawdown...")
fig, ax = plt.subplots(figsize=(12, 6), dpi=300)

def calc_drawdown(series):
    cum_max = series.cummax()
    return (series - cum_max) / cum_max * 100

for t, col in zip(['AAPL', 'NVDA', 'MSFT', 'AMZN'], [C_SLATE, C_CORAL, C_TEAL, C_GOLD]):
    sub = df_clean_3var[df_clean_3var['Ticker'] == t].set_index('Date').sort_index()
    if len(sub) > 0:
        dd = calc_drawdown(sub['Close'])
        ax.plot(dd.index, dd.values, label=f'{t} (Max DD: {dd.min():.1f}%)', color=col, linewidth=1.5, alpha=0.85)

ax.axhline(0, color='black', linewidth=1)
ax.fill_between(sub.index, -60, 0, color='gray', alpha=0.08)
ax.set_title("BIẾN ĐỘNG SỤT GIẢM TỐI ĐA TỪ ĐỈNH (UNDERWATER DRAWDOWN ANALYSIS 2021 - 2026)", fontsize=13, fontweight='bold', pad=15, color=C_NAVY)
ax.set_ylabel("Mức sụt giảm từ đỉnh giá (Drawdown %)", fontsize=11, fontweight='bold')
ax.set_xlabel("Thời gian giao dịch", fontsize=11, fontweight='bold')
ax.yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:.0f}%'))
ax.legend(loc='lower left', frameon=True)
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
p6 = os.path.join(CHARTS_DIR, "06_drawdown_analysis.png")
plt.savefig(p6)
plt.close()

# ------------------------------------------------------------------------------
# BIỂU ĐỒ 7: MA TRẬN NHIỆT TƯƠNG QUAN LỢI SUẤT (CORRELATION HEATMAP)
# ------------------------------------------------------------------------------
print("📊 [7/7] Xuất Biểu đồ 7: Ma trận nhiệt tương quan lợi suất giữa các ngành...")
sector_tickers = ['AAPL', 'MSFT', 'NVDA', 'JPM', 'BAC', 'JNJ', 'PFE', 'XOM', 'AMZN', 'PG']
pivoted_returns = df_clean_3var[df_clean_3var['Ticker'].isin(sector_tickers)].pivot(index='Date', columns='Ticker', values='Log_Return')
corr_matrix = pivoted_returns.corr()

fig, ax = plt.subplots(figsize=(9, 7.5), dpi=300)
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='Blues', vmin=0.1, vmax=0.8,
            square=True, linewidths=.8, cbar_kws={"shrink": .8, 'label': 'Hệ số tương quan Pearson (r)'}, ax=ax)

ax.set_title("MA TRẬN TƯƠNG QUAN TỶ SUẤT SINH LỢI GIỮA CÁC CỔ PHIẾU ĐẦU NGÀNH", fontsize=13, fontweight='bold', pad=15, color=C_NAVY)
plt.tight_layout()
p7 = os.path.join(CHARTS_DIR, "07_correlation_heatmap.png")
plt.savefig(p7)
plt.close()

print("\n" + "="*85)
print("✅ HOÀN TẤT XUẤT TOÀN BỘ 7 BIỂU ĐỒ TRỰC QUAN HÓA CHUẨN XUẤT BẢN!")
print(f"📁 Thư mục lưu trữ: {CHARTS_DIR}")
print("="*85)
